# Week 1, Lab 2 — Structured output (JSON you can trust)

**Course:** Agentic AI Engineering — Local Models Edition

Small local models love to add extra prose. Agents cannot. This lab teaches the pattern every later framework uses: **schema → prompt → parse → retry**.


## 1. Setup


In [ ]:
WEEK = 'Week 1'
LAB = 'Lab 2 — structured output'

import sys
from pathlib import Path

def _course_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "shared" / "course_runtime.py").exists():
            return p
    for c in [
        here / "agentic_ai_local",
        Path("/content/agentic_ai_local"),
        Path("/content"),
    ]:
        if (c / "shared" / "course_runtime.py").exists():
            return c
    return here

ROOT = _course_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shared.course_runtime import (
    detect_backend,
    print_banner,
    local_chat,
    calculator,
    lookup_fact,
    today_date,
    extract_json_object,
    parse_tool_call,
    openai_client_kwargs,
    get_langchain_llm,
    TOOL_SCHEMAS,
    MOCK_KB,
)

BACKEND = print_banner(WEEK, LAB)
print("If import failed, unzip/clone the WHOLE course folder (not a single notebook).")


In [ ]:
if BACKEND == "huggingface":
    %pip install -q transformers torch accelerate fastapi uvicorn pydantic
else:
    %pip install -q ollama pydantic


## 2. Why free text is not enough

Ask the model for a classification. You will often get a paragraph, markdown, or extra keys. Downstream Python code then crashes.


In [ ]:
from pydantic import BaseModel, Field, ValidationError

raw = local_chat([
    {"role": "user", "content": "Extract name, age, and city from: 'Maya is 29 and lives in Pune.'"}
], max_new_tokens=80)
print(raw)


## 3. Define the contract with Pydantic

Pydantic is the schema. The model does not 'know' Pydantic — **your loop** enforces it.


In [ ]:
class Person(BaseModel):
    name: str
    age: int = Field(ge=0, le=120)
    city: str

print(Person.model_json_schema())


## 4. Constrained prompt + parse + retry


In [ ]:
SCHEMA_PROMPT = """Return ONLY valid JSON matching this schema, no markdown:
{"name": string, "age": integer, "city": string}

Text: {text}
"""

def structured_person(text: str, attempts: int = 3) -> Person:
    messages = [{"role": "user", "content": SCHEMA_PROMPT.format(text=text)}]
    last_err = None
    for i in range(attempts):
        reply = local_chat(messages, max_new_tokens=80, temperature=0.1)
        obj = extract_json_object(reply)
        try:
            if not obj:
                raise ValueError(f"no JSON in: {reply!r}")
            return Person.model_validate(obj)
        except (ValidationError, ValueError) as err:
            last_err = err
            messages.append({"role": "assistant", "content": reply})
            messages.append({
                "role": "user",
                "content": f"That was invalid ({err}). Return ONLY the JSON object.",
            })
            print(f"retry {i+1}: {err}")
    raise RuntimeError(f"failed after {attempts} attempts: {last_err}")

person = structured_person("Maya is 29 and lives in Pune.")
print(person)
print(person.model_dump())


## 5. Exercise

1. Add a `sentiment` field (`positive` | `negative` | `neutral`) and classify: *The lab was hard but I learned a lot.*
2. Feed garbage text (`asdf`) and watch retries fail — then add a fallback `Person(name='unknown', age=0, city='unknown')`.
3. Compare `temperature=0` vs `0.9` for JSON reliability.

**Next:** `lab3_tools_from_scratch.ipynb` — the model chooses a Python function to run.
